In [1]:
import json
import requests

# 1. 내가 찾은 원티드 API 주소 명시 (Endpoint 및 Parameter 분리)
URL = "https://www.wanted.co.kr/api/chaos/navigation/v1/results"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*"
}
PARAMS = {
    "job_group_id": 518,        # 개발 전체 직군 코드
    "country": "kr",            # 한국 공고
    "job_sort": "job.latest_order", # 최신순 정렬
    "years": 0,                 # 경력 제한 없음
    "locations": "all",
    "limit": 5,                 # 규격서 확인용이므로 샘플로 5개만 요청
    "offset": 0                 # 첫 페이지 시작 인덱스
}

print("🔍 [WBS 1단계] 원티드 공고 조회 API 명세 검증을 시작합니다...")

try:
    # 2. API 호출
    response = requests.get(URL, headers=HEADERS, params=PARAMS, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        
        # ==================================================
        # [확인 1] 서버가 돌려준 전체 응답 구조(JSON)의 최상위 키 확인
        # ==================================================
        print("\n==========================================")
        print("📊 [응답 구조 데이터 규격 필드 (Top-level Keys)]")
        print("==========================================")
        print(f"-> 서버 응답 결과 포함된 키 목록: {list(data.keys())}")
        
        # ==================================================
        # [확인 2] 실제 공고 리스트가 담긴 'data' 필드 내부 쪼개기
        # ==================================================
        job_list = data.get("data", [])
        print(f"-> 요청한 샘플 공고 개수: {len(job_list)}개")
        
        if job_list:
            sample_job = job_list[0]
            print("\n==========================================")
            print("📋 [단일 공고 데이터 내부 필드 명세 샘플]")
            print("==========================================")
            # 1단계 규격 정의서 작성을 위해 단일 데이터 구조를 이쁘게 출력
            print(json.dumps(sample_job, indent=4, ensure_ascii=False)[:1200]) 
            print("\n(...하략...)")
            
            # ==================================================
            # [확인 3] 우리가 추출해야 할 핵심 데이터 매핑 검증
            # ==================================================
            print("\n==========================================")
            print("🎯 [최종 규격서 작성을 위한 핵심 항목 매핑 결과]")
            print("==========================================")
            for idx, job in enumerate(job_list, 1):
                job_id = job.get("id")
                company_name = job.get("company", {}).get("name")
                position_title = job.get("position")
                
                # 기술 스택이 들어있는 태그 정보 추출 테스트
                tags = [tag.get("name") for tag in job.get("tags", [])]
                
                print(f"[{idx}] 공고 ID: {job_id}")
                print(f"    회사명 : {company_name}")
                print(f"    공고명 : {position_title}")
                print(f"    태그 정보(기술 키워드 후보군): {tags}")
                print("-" * 40)
                
    else:
        print(f"❌ API 호출 실패 (HTTP 상태 코드: {response.status_code})")

except Exception as e:
    print(f"💥 네트워크 연결 또는 파싱 에러 발생: {e}")

🔍 [WBS 1단계] 원티드 공고 조회 API 명세 검증을 시작합니다...

📊 [응답 구조 데이터 규격 필드 (Top-level Keys)]
-> 서버 응답 결과 포함된 키 목록: ['data', 'links']
-> 요청한 샘플 공고 개수: 5개

📋 [단일 공고 데이터 내부 필드 명세 샘플]
{
    "id": 323359,
    "reward_total": "100만원",
    "is_bookmark": false,
    "company": {
        "id": 1235,
        "name": "라이너(Liner)",
        "application_response_stats": {
            "avg_rate": 0.0,
            "level": "very_low"
        }
    },
    "title_img": {
        "origin": "https://static.wanted.co.kr/images/company/1235/v3mgc8u1new18nxw__1080_790.jpg",
        "thumb": "https://static.wanted.co.kr/images/company/1235/v3mgc8u1new18nxw__400_400.jpg",
        "video": null
    },
    "address": {
        "country": "한국",
        "location": "서울",
        "district": "마포구"
    },
    "position": "ML Engineer - 전문연구요원(신규 편입/전직)",
    "category_tag": {
        "parent_id": 518,
        "id": 1634
    },
    "attraction_tags": [
        10433,
        10402,
        10436,
        10437,
        10468,
  

In [2]:
import time
import random
import pandas as pd
import requests

# ==========================================
# 1. 글로벌 설정 및 API 엔드포인트 정의
# ==========================================
LIST_URL = "https://www.wanted.co.kr/api/chaos/navigation/v1/results"
DETAIL_URL_TEMPLATE = "https://www.wanted.co.kr/api/chaos/jobs/v1/{job_id}/details"
OUTPUT_FILE = "wanted_raw_id_dataset_refined.csv" # 저장용 파일명

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    "Origin": "https://www.wanted.co.kr",
    "Referer": "https://www.wanted.co.kr/"
}

def get_bulk_job_ids(job_group_id=518, max_pages=30):
    """지정된 페이지 수만큼 목록 API를 순회하여 공고 ID 풀을 확보합니다."""
    print(f"📢 [Phase 1] 공고 목록 조회 시작 (목표: {max_pages} 페이지)...")
    job_ids_pool = []
    limit = 20
    
    for page in range(max_pages):
        offset = page * limit
        params = {
            "job_group_id": job_group_id,
            "country": "kr",
            "job_sort": "job.latest_order",
            "years": "0",
            "locations": "all",
            "limit": limit,
            "offset": offset
        }
        try:
            response = requests.get(LIST_URL, headers=HEADERS, params=params, timeout=10)
            if response.status_code == 200:
                data = response.json().get("data", [])
                if not data: break
                for item in data:
                    job_ids_pool.append({
                        "job_id": item.get("id"),
                        "company": item.get("company", {}).get("name"),
                        "position": item.get("position")
                    })
            else: break
        except Exception as e:
            print(f"   💥 예외 발생: {e}")
            break
        time.sleep(random.uniform(0.5, 1.0))
    return job_ids_pool

# ==========================================
# 2. 2단계: 상세 API 호출 및 항목별 분리 정제 추출
# ==========================================
def fetch_bulk_job_details_separated(job_list):
    """
    공고 상세 API에서 자격요건과 우대사항을 믹스하지 않고
    각각 개별적인 데이터 필드로 완전 분리하여 수집 데이터프레임 구조를 형성합니다.
    """
    print(f"\n📢 [Phase 2] 총 {len(job_list)}개 공고 대상 항목별(자격요건 vs 우대사항) 분리 적재 시작...")
    
    refined_dataset = []
    total_requests = len(job_list)
    
    for idx, base_job in enumerate(job_list, 1):
        job_id = base_job["job_id"]
        company = base_job["company"]
        position = base_job["position"]
        
        detail_url = DETAIL_URL_TEMPLATE.format(job_id=job_id)
        
        try:
            response = requests.get(detail_url, headers=HEADERS, timeout=10)
            
            if response.status_code == 200:
                detail_data = response.json().get("job", {})
                job_detail = detail_data.get("detail", {})
                
                # 🎯 [구분 적재 구조 변경] 단일 텍스트로 합치지 않고 각각 원천 데이터 유지
                requirements = job_detail.get("requirements", "").strip()      # 자격 요건
                preferred = job_detail.get("preferred_points", "").strip()     # 우대 사항
                
                refined_dataset.append({
                    "job_id": job_id,
                    "company": company,
                    "position": position,
                    "requirements": requirements, # 컬럼 분리 1
                    "preferred": preferred        # 컬럼 분리 2
                })
                print(f"   🚀 [{idx}/{total_requests}] 분리 성공: {company} - {position}")
                
            elif response.status_code == 429:
                print(f"   🛑 [{idx}/{total_requests}] 차단 감지 (HTTP 429)")
                
        except Exception as e:
            print(f"   💥 [{idx}/{total_requests}] 예외 발생: {e}")
            
        time.sleep(random.uniform(1.5, 2.5))
        
        # 50개 단위 중간 백업 시에도 분리된 구조 유지
        if idx % 50 == 0 and refined_dataset:
            pd.DataFrame(refined_dataset).to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
            
    return refined_dataset

if __name__ == "__main__":
    TARGET_PAGES = 30 
    bulk_job_pool = get_bulk_job_ids(job_group_id=518, max_pages=TARGET_PAGES)
    
    if bulk_job_pool:
        final_dataset = fetch_bulk_job_details_separated(bulk_job_pool)
        if final_dataset:
            df = pd.DataFrame(final_dataset)
            df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
            print(f"\n🎉 [수정 완료] 자격요건과 우대사항이 완벽히 분리된 파일이 생성되었습니다: '{OUTPUT_FILE}'")

📢 [Phase 1] 공고 목록 조회 시작 (목표: 30 페이지)...

📢 [Phase 2] 총 409개 공고 대상 항목별(자격요건 vs 우대사항) 분리 적재 시작...
   🚀 [1/409] 분리 성공: 라이너(Liner) - ML Engineer - 전문연구요원(신규 편입/전직)
   🚀 [2/409] 분리 성공: 에이티씨아이 - 시니어 백엔드 엔지니어 (Django, FastAPI, Python)
   🚀 [3/409] 분리 성공: 바카티오(Vacatio) - [인턴] Backend Engineer
   🚀 [4/409] 분리 성공: 바카티오(Vacatio) - Backend Engineer
   🚀 [5/409] 분리 성공: 바카티오(Vacatio) - Frontend Engineer
   🚀 [6/409] 분리 성공: 뉴로다임 - 파이썬 프로그램 개발자
   🚀 [7/409] 분리 성공: 문토 - 풀스택 개발자
   🚀 [8/409] 분리 성공: 미리디 - [병역특례] 미리캔버스 에디터 엔지니어 [전문연구요원]
   🚀 [9/409] 분리 성공: 페이타랩(패스오더) - DevOps Engineer
   🚀 [10/409] 분리 성공: 페이타랩(패스오더) - [주니어] 데이터 엔지니어 (Growth Data Engineer), 서울
   🚀 [11/409] 분리 성공: 피에프씨테크놀로지스 - [인턴] ML Engineer
   🚀 [12/409] 분리 성공: 소크라에이아이 - AI Research Scientist (전문연구요원 가능)
   🚀 [13/409] 분리 성공: 유에스소프트 - RFID 주니어 개발자 (3년 이하)
   🚀 [14/409] 분리 성공: 캐리마텍 - 3D프린터 제어 프로그램 개발자
   🚀 [15/409] 분리 성공: 팀엘리시움 - [전문연구요원] 디지털 헬스케어 스타트업 AI/Computer Vision 연구원
   🚀 [16/409] 분리 성공: 에너자이(ENERZAi) - (병역특례 가능) AI 최적화 Researcher